In [ ]:
import pandas as pd

# load data
df = pd.read_csv('../data/application_train.csv')

# basic shape
df.shape

In [ ]:
target_count=df['TARGET'].value_counts()
target_ratio=df['TARGET'].value_counts(normalize=True)
print(target_count)
print(target_ratio)

“The dataset is imbalanced with a relatively low default rate, which is typical in credit portfolios.”

In [ ]:
df.head()

In [ ]:
summary = pd.DataFrame({
    'column': df.columns,
    'dtype': df.dtypes.values,
    'non_null_count': df.notnull().sum().values,
    'missing_pct': (df.isnull().mean().values * 100).round(2)
})

summary.head()

In [ ]:
dtype_group = summary.groupby('dtype')['column'].apply(list)

for dtype, cols in dtype_group.items():
    print(f"\n{dtype} ({len(cols)} columns):")
    print(cols[:10]) 

In [ ]:
numeric_df = summary[summary['dtype'] != 'object']
categorical_df = summary[summary['dtype'] == 'object']

print(numeric_df.head())
print(categorical_df.head())

In [ ]:
missing = df.isnull().mean().sort_values(ascending=False)
missing_pct = (missing * 100).round(2)
missing_pct.head(20)

In [ ]:
missing_table = pd.DataFrame({
    'column': missing.index,
    'missing_pct': (missing.values * 100).round(2)
})

missing_table.head(20)

In [ ]:
#drop columns with >50% missing
cols_to_drop = missing[missing > 0.5].index
len(cols_to_drop)

In [ ]:
df_clean = df.drop(columns=cols_to_drop)
df_clean.shape

In [ ]:
good_features = summary[summary['missing_pct'] < 30]
good_features = good_features.reset_index(drop=True)
print(len(good_features))

In [ ]:
import pandas as pd

pd.set_option('display.max_rows', None)

good_features[['column']].sort_values(by='column')

In [ ]:
base_features = [
    'AMT_INCOME_TOTAL',
    'AMT_CREDIT',
    'AMT_ANNUITY',
    'AMT_GOODS_PRICE',
    'DAYS_BIRTH',
    'DAYS_EMPLOYED',
    'CNT_CHILDREN'
]


In [ ]:
import numpy as np
import pandas as pd

# rebuild from original df, not the already-damaged object column
df_clean['DAYS_EMPLOYED'] = pd.to_numeric(df['DAYS_EMPLOYED'], errors='coerce')
df_clean['DAYS_EMPLOYED'] = df_clean['DAYS_EMPLOYED'].replace(365243, np.nan)

# check type
print(df_clean['DAYS_EMPLOYED'].dtype)

# numeric summary
df_clean['DAYS_EMPLOYED'].describe()

In [ ]:
#Debt-to-Income
df_clean['DTI'] = df_clean['AMT_CREDIT'] / df_clean['AMT_INCOME_TOTAL']

#Installment burden
df_clean['INST_TO_INCOME'] = df_clean['AMT_ANNUITY'] / df_clean['AMT_INCOME_TOTAL']

#Loan-to-value proxy
df_clean['LTV_PROXY'] = df_clean['AMT_CREDIT'] / df_clean['AMT_GOODS_PRICE']

#Age
df_clean['AGE'] = -df_clean['DAYS_BIRTH'] / 365

#Employment years
df_clean['EMPLOYMENT_YEARS'] = -df_clean['DAYS_EMPLOYED'] / 365

In [ ]:
features = [
    'AMT_INCOME_TOTAL',
    'AMT_CREDIT',
    'AMT_ANNUITY',
    'DTI',
    'INST_TO_INCOME',
    'LTV_PROXY',
    'AGE',
    'EMPLOYMENT_YEARS',
    'CNT_CHILDREN'
]

target = 'TARGET'

df_model = df_clean[features + [target]].dropna()

print(df_model.shape)
print(df_model.describe())

In [ ]:
X = df_model.drop(columns=['TARGET'])
y = df_model['TARGET']

In [ ]:
pip install scikit-learn

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Build logistic regression model

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(X_train_scaled, y_train)

In [ ]:
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

In [ ]:
from sklearn.metrics import roc_auc_score

auc = roc_auc_score(y_test, y_pred_proba)
print("AUC:", auc)

In [ ]:
pip install matplotlib

In [ ]:
from sklearn.metrics import roc_curve
import matplotlib.pyplot as plt

fpr, tpr, _ = roc_curve(y_test, y_pred_proba)

plt.figure()
plt.plot(fpr, tpr)
plt.plot([0,1], [0,1], linestyle='--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.show()

The model shows moderate discriminatory power with an AUC of ~0.62, indicating it performs better than random but has room for improvement. This is expected given the limited feature set.

In [ ]:
features_1 = [
    'AMT_INCOME_TOTAL',
    'AMT_CREDIT',
    'AMT_ANNUITY',
    'DTI',
    'INST_TO_INCOME',
    'LTV_PROXY',
    'AGE',
    'EMPLOYMENT_YEARS',
    'CNT_CHILDREN',
    'EXT_SOURCE_2',
    'EXT_SOURCE_3'
]

target = 'TARGET'

# Build model dataset first
df_model = df_clean[features_1 + [target]].copy()

# Identify numeric columns in df_model
numeric_cols = df_model.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Remove target from imputation columns
if target in numeric_cols:
    numeric_cols.remove(target)

# Median imputation for numeric feature columns
df_model[numeric_cols] = df_model[numeric_cols].fillna(df_model[numeric_cols].median())

# Check result
print(df_model.shape)
print(df_model.describe())
print(df_model.isnull().sum())

Correlation filtering

In [ ]:
import matplotlib.pyplot as plt

corr_matrix = df_model[features_1].corr().abs()

plt.figure(figsize=(10, 8))
plt.imshow(corr_matrix, interpolation='nearest')
plt.colorbar()
plt.xticks(range(len(corr_matrix.columns)), corr_matrix.columns, rotation=90)
plt.yticks(range(len(corr_matrix.columns)), corr_matrix.columns)
plt.title("Absolute Correlation Matrix")
plt.tight_layout()
plt.show()

In [ ]:
threshold = 0.80

upper_triangle = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

high_corr_pairs = []

for col in upper_triangle.columns:
    for row in upper_triangle.index:
        corr_value = upper_triangle.loc[row, col]
        if pd.notnull(corr_value) and corr_value > threshold:
            high_corr_pairs.append((row, col, corr_value))

high_corr_df = pd.DataFrame(high_corr_pairs, columns=['feature_1', 'feature_2', 'abs_corr'])
high_corr_df.sort_values(by='abs_corr', ascending=False)
print(high_corr_df)

to_drop_corr = [col for col in upper_triangle.columns if any(upper_triangle[col] > threshold)]

print("Features to drop from correlation filter:")
print(to_drop_corr)

features_after_corr = [col for col in candidate_features if col not in to_drop_corr]

print("\nFeatures kept after correlation filtering:")
for col in features_after_corr:
    print(col)

print("\nNumber kept:", len(features_after_corr))

In [ ]:
X = df_model[features_1].copy().dropna()
y = df_model[target].copy()

print(X.shape, y.shape)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
from sklearn.feature_selection import RFECV
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold

log_reg = LogisticRegression(max_iter=1000, solver='liblinear')

rfecv = RFECV(
    estimator=log_reg,
    step=1,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='roc_auc',
    min_features_to_select=3,
    n_jobs=-1
)

rfecv.fit(X_train_scaled, y_train)

In [ ]:
selected_features = X.columns[rfecv.support_].tolist()

print("Optimal number of features:", rfecv.n_features_)
print("\nSelected features:")
for col in selected_features:
    print(col)

In [ ]:
rfecv_results = pd.DataFrame({
    'feature': X.columns,
    'selected': rfecv.support_,
    'ranking': rfecv.ranking_
}).sort_values(by=['selected', 'ranking'], ascending=[False, True])

rfecv_results

In [ ]:
rfecv_scores = pd.DataFrame({
    'num_features': range(1, len(rfecv.cv_results_['mean_test_score']) + 1),
    'mean_auc': rfecv.cv_results_['mean_test_score']
})

rfecv_scores

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(rfecv_scores['num_features'], rfecv_scores['mean_auc'], marker='o')
plt.xlabel("Number of Features Selected")
plt.ylabel("Cross-Validated AUC")
plt.title("RFECV Performance")
plt.grid(True)
plt.show()

Using RFECV, all candidate features were retained, indicating each variable contributed incremental predictive power. The model shows stable performance with minimal multicollinearity.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, roc_curve
import pandas as pd
import matplotlib.pyplot as plt

# selected features from RFECV
selected_features = X.columns[rfecv.support_].tolist()

print("Selected features:")
for col in selected_features:
    print(col)

# build final train/test sets
X_train_final = X_train[selected_features].copy()
X_test_final = X_test[selected_features].copy()

# scale
scaler_final = StandardScaler()
X_train_final_scaled = scaler_final.fit_transform(X_train_final)
X_test_final_scaled = scaler_final.transform(X_test_final)

# fit final logistic regression
final_model = LogisticRegression(max_iter=1000, solver='liblinear')
final_model.fit(X_train_final_scaled, y_train)

# predict PD
y_pred_proba_final = final_model.predict_proba(X_test_final_scaled)[:, 1]

# AUC
final_auc = roc_auc_score(y_test, y_pred_proba_final)
print("Final AUC:", final_auc)

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score
import matplotlib.pyplot as plt

# ROC values
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba_final)

# AUC
auc_value = roc_auc_score(y_test, y_pred_proba_final)

# Plot
plt.figure(figsize=(7, 6))

plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {auc_value:.3f})', linewidth=2)
plt.plot([0, 1], [0, 1], linestyle='--', linewidth=2, label='Random Model')

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC Curve)')
plt.legend(loc='lower right')
plt.grid(True)

plt.show()

The final model achieved an AUC of 0.72, indicating good discriminatory power for a baseline PD model built using application-level data, with potential to improve further through bureau and behavioural features.”

In [ ]:
import pandas as pd

coef_table = pd.DataFrame({
    'feature': selected_features,
    'coefficient': final_model.coef_[0]
})

coef_table['direction'] = coef_table['coefficient'].apply(
    lambda x: 'Higher default risk' if x > 0 else 'Lower default risk'
)

coef_table

In [ ]:
odd_ratio=coef_table['odds_ratio'] = np.exp(coef_table['coefficient'])
odd_ratio

coefficient > 0 → increases default risk
coefficient < 0 → reduces default risk
odds_ratio > 1 → higher odds of default
odds_ratio < 1 → lower odds of default

In [ ]:
for _, row in coef_table.iterrows():
    print(f"Feature: {row['feature']}")
    print(f"Coefficient: {row['coefficient']:.4f}")
    print(f"Odds Ratio: {row['odds_ratio']:.4f}")
    print(f"Direction: {row['direction']}")
    print("-" * 50)

Model suggests default risk is mainly driven by:

Repayment burden (VERY important)
Leverage / collateral (VERY important)
External credit score (MOST important)
Stability (age + employment)

Key Drivers
1. Repayment burden → higher risk
INST_TO_INCOME (strongest positive driver)
AMT_ANNUITY

Borrowers who spend a larger portion of income on repayments are more likely to default

2. Leverage / collateral → higher risk
LTV_PROXY: Higher loan relative to asset value = higher default risk

3. External credit score → lower risk (STRONGEST effect)
EXT_SOURCE_2
EXT_SOURCE_3

Borrowers with better external credit scores are significantly less likely to default

4. Stability → lower risk
AGE
EMPLOYMENT_YEARS

Older borrowers and those with longer employment history are more stable and less risky

5. Income & loan size (mixed signals)
AMT_INCOME_TOTAL → slightly increases risk (unexpected)
AMT_CREDIT → slightly reduces risk

These variables alone are not strong drivers — risk is better captured by ratios (like DTI, installment burden)

6. Weak / negligible variables
CNT_CHILDREN
DTI (surprisingly weak in above model)

These have very small impact and could potentially be removed without much performance loss

summary:

The model shows that default risk is primarily driven by repayment burden and leverage, with higher installment-to-income and loan-to-value ratios increasing risk. External credit scores are the strongest predictors, significantly reducing default probability. Borrower stability, reflected by age and employment history, also lowers risk. Overall, the model aligns well with credit risk intuition, with behavioural and affordability metrics being more predictive than absolute income or loan size.